# Oelpreis-Datenanalyse: Bereinigung & Vergleich mit Forex

**Ziel:** Oelpreisdaten (WTI, Brent) aus Yahoo Finance analysieren, Datenqualitaet pruefen und Korrelation mit Forex-Kursen (EUR/USD, EUR/CHF, GBP/USD) untersuchen.

**Datenquellen:** Yahoo Finance (Oel + Forex)

---

## 1. Setup & Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (14, 6)
pd.set_option("display.max_columns", None)

OIL_DIR = os.path.join("..", "..", "data", "raw", "oil", "yahoo")
FOREX_DIR = os.path.join("..", "..", "data", "raw", "forex")
OIL_TICKERS = ["WTI_Crude_Oil", "Brent_Crude_Oil"]
FOREX_PAIRS = ["EUR_USD", "EUR_CHF", "GBP_USD"]

print("Setup erfolgreich!")

In [ ]:
# Oeldaten laden und normalisieren
def load_oil_yahoo(ticker):
    """Yahoo Finance Oil CSV laden und normalisieren."""
    pattern = os.path.join(OIL_DIR, f"{ticker}_*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"FEHLER: Keine Datei gefunden fuer {ticker} in {OIL_DIR}")
    df = pd.read_csv(files[-1], index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
    df.index = df.index.ceil("D")
    df.index.name = "date"
    df = df.rename(columns=str.lower)
    df = df[~df.index.duplicated(keep="first")]
    return df[["open", "high", "low", "close", "volume"]].copy()

def load_forex_yahoo(pair):
    """Yahoo Finance Forex CSV laden und normalisieren."""
    pattern = os.path.join(FOREX_DIR, "yahoo", f"{pair}_*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"FEHLER: Keine Datei gefunden fuer {pair}")
    df = pd.read_csv(files[-1], index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
    df.index = df.index.ceil("D")
    df.index.name = "date"
    df = df.rename(columns=str.lower)
    df = df[~df.index.duplicated(keep="first")]
    return df[["open", "high", "low", "close"]].copy()

# Oeldaten laden
oil_data = {}
for ticker in OIL_TICKERS:
    oil_data[ticker] = load_oil_yahoo(ticker)

# Forex-Daten laden (Yahoo als Referenz)
forex_data = {}
for pair in FOREX_PAIRS:
    forex_data[pair] = load_forex_yahoo(pair)

# Uebersicht
print("=== Oeldaten ===")
for ticker, df in oil_data.items():
    weekends = (df.index.weekday >= 5).sum()
    print(f"  {ticker:20s}: {len(df):5d} Zeilen, {df.index.min().date()} bis {df.index.max().date()}, Wochenenden: {weekends}")

print("\n=== Forex-Daten (Yahoo) ===")
for pair, df in forex_data.items():
    weekends = (df.index.weekday >= 5).sum()
    print(f"  {pair:20s}: {len(df):5d} Zeilen, {df.index.min().date()} bis {df.index.max().date()}, Wochenenden: {weekends}")

## 2. Duplikate pruefen

Gibt es doppelte Datumseintraege?

In [ ]:
# 2.1 Duplikate im Index (doppelte Tage)
print("=== Duplikate im Datumsindex ===\n")
for ticker, df in oil_data.items():
    n_dupes = df.index.duplicated().sum()
    print(f"  {ticker}: {n_dupes} Duplikate")

# 2.2 Inhaltliche Duplikate (identische OHLC-Werte an verschiedenen Tagen)
print("\n=== Inhaltliche Duplikate (identische OHLC-Werte an verschiedenen Tagen) ===\n")
for ticker, df in oil_data.items():
    n_content_dupes = df[["open", "high", "low", "close"]].duplicated().sum()
    print(f"  {ticker}: {n_content_dupes} inhaltliche Duplikate")

## 3. Fehlende Tage & NaN-Werte

In [ ]:
# 3.1 Wochentage vs. Wochenenden
print("=== Wochentage vs. Wochenenden ===\n")
for ticker, df in oil_data.items():
    weekdays = (df.index.weekday < 5).sum()
    weekends = (df.index.weekday >= 5).sum()
    print(f"  {ticker}: {weekdays} Wochentage, {weekends} Wochenenden")

# 3.2 Fehlende Werte in OHLC-Spalten
print("\n=== Fehlende Werte (NaN) in OHLC-Spalten ===\n")
for ticker, df in oil_data.items():
    missing = df[["open", "high", "low", "close"]].isnull().sum()
    if missing.sum() == 0:
        print(f"  {ticker}: Keine fehlenden Werte")
    else:
        print(f"  {ticker}:")
        for col, n in missing.items():
            if n > 0:
                print(f"    {col}: {n} fehlend")

In [ ]:
# 3.3 Datenabdeckung: Vergleich WTI vs. Brent
wti_dates = set(oil_data["WTI_Crude_Oil"].index)
brent_dates = set(oil_data["Brent_Crude_Oil"].index)

only_wti = wti_dates - brent_dates
only_brent = brent_dates - wti_dates
common = wti_dates & brent_dates

print("=== Datumsvergleich WTI vs. Brent ===\n")
print(f"  Gemeinsame Tage:   {len(common)}")
print(f"  Nur WTI:           {len(only_wti)}")
print(f"  Nur Brent:         {len(only_brent)}")

if only_wti:
    print(f"\n  Tage nur in WTI:")
    for d in sorted(only_wti)[:10]:
        print(f"    {d.strftime('%Y-%m-%d')} ({['Mo','Di','Mi','Do','Fr','Sa','So'][d.weekday()]})")

if only_brent:
    print(f"\n  Tage nur in Brent:")
    for d in sorted(only_brent)[:10]:
        print(f"    {d.strftime('%Y-%m-%d')} ({['Mo','Di','Mi','Do','Fr','Sa','So'][d.weekday()]})")

## 4. Preisvergleich WTI vs. Brent

Wie stark korrelieren die beiden Oelsorten? Die Preisdifferenz (Brent-WTI Spread) ist ein wichtiger Marktindikator.

In [ ]:
# 4.1 Close-Preise zusammenfuehren
oil_close = pd.DataFrame({
    "WTI": oil_data["WTI_Crude_Oil"]["close"],
    "Brent": oil_data["Brent_Crude_Oil"]["close"],
}).dropna()

oil_close["Spread_Brent_WTI"] = oil_close["Brent"] - oil_close["WTI"]
oil_close["Spread_pct"] = (oil_close["Spread_Brent_WTI"] / oil_close["WTI"] * 100).round(2)

print(f"Gemeinsame Handelstage: {len(oil_close)}")
print(f"\nSpread-Statistik (Brent - WTI):")
display(oil_close["Spread_Brent_WTI"].describe().round(2))

print(f"\nKorrelation WTI vs. Brent (Close): {oil_close['WTI'].corr(oil_close['Brent']):.6f}")

In [ ]:
# 4.2 Visualisierung: WTI vs. Brent Kursverlauf und Spread
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(oil_close.index, oil_close["WTI"], label="WTI Crude Oil", linewidth=1)
ax1.plot(oil_close.index, oil_close["Brent"], label="Brent Crude Oil", linewidth=1)
ax1.set_title("Oelpreise: WTI vs. Brent (USD/Barrel)", fontsize=14)
ax1.set_ylabel("Preis (USD)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(oil_close.index, oil_close["Spread_Brent_WTI"], alpha=0.4, label="Brent-WTI Spread")
ax2.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
ax2.axhline(y=oil_close["Spread_Brent_WTI"].mean(), color="red", linestyle="--", linewidth=0.8, 
            label=f"Mittelwert: ${oil_close['Spread_Brent_WTI'].mean():.2f}")
ax2.set_title("Brent-WTI Spread (USD)", fontsize=14)
ax2.set_ylabel("Spread (USD)")
ax2.set_xlabel("Datum")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Korrelation Oel vs. Forex

Zentraler Analysepunkt: Wie stark korrelieren Oelpreise mit Waehrungskursen? 
Wir berechnen die Korrelation sowohl auf Preisniveau als auch auf Basis taeglicher Renditen.

In [ ]:
# 5.1 Alle Close-Preise in einem DataFrame zusammenfuehren
all_close = pd.DataFrame()

for ticker, df in oil_data.items():
    all_close[ticker] = df["close"]

for pair, df in forex_data.items():
    all_close[pair] = df["close"]

# Nur gemeinsame Tage (inner join)
all_close = all_close.dropna()
print(f"Gemeinsame Handelstage (alle Instrumente): {len(all_close)}")
print(f"Zeitraum: {all_close.index.min().date()} bis {all_close.index.max().date()}")
display(all_close.head())

In [ ]:
# 5.2 Korrelationsmatrix auf Preisniveau
print("=== Korrelationsmatrix (Close-Preise) ===\n")
corr_prices = all_close.corr().round(4)
display(corr_prices)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_prices, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, fmt=".3f")
ax.set_title("Korrelationsmatrix: Close-Preise (Oel & Forex)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Korrelationsmatrix auf Rendite-Basis (aussagekraeftiger als Preisniveau)
returns = all_close.pct_change().dropna()

print("=== Korrelationsmatrix (taegliche Renditen) ===\n")
corr_returns = returns.corr().round(4)
display(corr_returns)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_returns, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, fmt=".3f")
ax.set_title("Korrelationsmatrix: Taegliche Renditen (Oel & Forex)", fontsize=14)
plt.tight_layout()
plt.show()

print("\nHinweis: Korrelation auf Rendite-Basis ist aussagekraeftiger als auf Preisniveau,")
print("da Preisniveau-Korrelationen oft durch gemeinsame Trends verzerrt sind (Spurious Correlation).")

## 6. Rollende Korrelation

Die Korrelation zwischen Oel und Forex ist nicht stabil ueber die Zeit. 
Eine rollende Korrelation (z.B. 30 Tage) zeigt, wie sich der Zusammenhang veraendert.

In [ ]:
# 6.1 Rollende 30-Tage-Korrelation: WTI vs. Forex-Paare (auf Rendite-Basis)
WINDOW = 30

fig, axes = plt.subplots(len(FOREX_PAIRS), 1, figsize=(16, 4 * len(FOREX_PAIRS)))

for ax, pair in zip(axes, FOREX_PAIRS):
    rolling_corr = returns["WTI_Crude_Oil"].rolling(WINDOW).corr(returns[pair])
    
    ax.plot(rolling_corr.index, rolling_corr, linewidth=1, label=f"WTI vs. {pair}")
    ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
    ax.set_title(f"Rollende {WINDOW}-Tage-Korrelation: WTI vs. {pair}", fontsize=12)
    ax.set_ylabel("Korrelation")
    ax.set_ylim(-1, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Datum")
plt.tight_layout()
plt.show()

## 7. Normalisierte Kurse ueberlagert

Alle Instrumente auf Indexbasis (Startwert = 100) normalisiert, um relative Entwicklungen vergleichbar zu machen.

In [ ]:
# 7.1 Normalisierte Kurse (Indexbasis 100)
normalized = (all_close / all_close.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(16, 8))

for col in normalized.columns:
    ax.plot(normalized.index, normalized[col], label=col, linewidth=1)

ax.axhline(y=100, color="black", linestyle="--", linewidth=0.5, alpha=0.5)
ax.set_title("Normalisierte Kursentwicklung (Indexbasis 100)", fontsize=14)
ax.set_xlabel("Datum")
ax.set_ylabel("Index (Start = 100)")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Kombiniertes CSV exportieren

Oel- und Forex-Daten werden in ein kombiniertes CSV fuer weitere Analysen exportiert.

In [ ]:
# 8.1 Kombiniertes DataFrame erstellen
PROCESSED_DIR = os.path.join("..", "..", "data", "processed", "oil")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Alle Oeldaten mit OHLC + Volume zusammenfuehren
all_oil = pd.DataFrame()
for ticker, df in oil_data.items():
    prefix = ticker.lower().replace(" ", "_")
    for col in ["open", "high", "low", "close", "volume"]:
        all_oil[f"{prefix}_{col}"] = df[col]

# Forex Close-Preise als Kontext hinzufuegen
for pair, df in forex_data.items():
    all_oil[f"{pair.lower()}_close"] = df["close"]

all_oil = all_oil.sort_index()
all_oil.index.name = "date"

# Metadaten
all_oil["weekday"] = all_oil.index.weekday
all_oil["is_weekend"] = all_oil["weekday"] >= 5

print(f"Kombiniertes DataFrame: {len(all_oil)} Zeilen, {all_oil.shape[1]} Spalten")
print(f"Spalten: {all_oil.columns.tolist()}")
display(all_oil.head())

In [ ]:
# 8.2 Als CSV speichern
output_path = os.path.join(PROCESSED_DIR, "oil_forex_kombiniert.csv")
all_oil.to_csv(output_path)
print(f"Gespeichert: {output_path}")
print(f"Groesse: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Zeilen: {len(all_oil)}, Spalten: {all_oil.shape[1]}")

# Preview
print(f"\nBeispiel (erste 10 Tage):")
display(all_oil.head(10))

## Erkenntnisse

**WTI vs. Brent:**
- (hier Ergebnisse eintragen nach Ausfuehrung)
- Spread-Analyse zeigt...

**Korrelation Oel vs. Forex:**
- (hier Ergebnisse eintragen)
- Rollende Korrelation zeigt zeitliche Instabilitaet...

**Naechste Schritte:**
1. Vertiefung der Korrelationsanalyse mit zeitversetzten Variablen (Lag-Analyse)
2. Integration mit Newsdaten (Sentiment) fuer multivariate Analyse
3. Untersuchung ob Oelpreis als Fruehindikator fuer Waehrungsbewegungen dient